## AI Text Detection ##

We run Pangram on all cleaned sampled texted across all sampled candidates. The final dataframe returns the given CSV with the additional columns of label_final, ai_assistance_final, confidence_final, fraction_ai, fraction_ai_assisted, fraction_human, and num_ai_segments; all of these scores are obtained with the Pangram API using research credits

In [69]:
import pandas as pd
from pangram import Pangram
import numpy as np
import random
import re
from collections import Counter #for extracting labels and confidence majorities with pangram
import os
from dotenv import load_dotenv

In [71]:
data = pd.read_csv("../content_export.csv")

In [73]:
load_dotenv()
pangram_client = Pangram(api_key=os.getenv("PANGRAM_API_KEY"))

### Data Cleaning ###
This notebook uses data queried from camplinks.db from the Content table to get the sampled text for all candidates in our database.This is loaded in as "data". We then clean the dataframe by dropping rows where the unprocessed_text and sampled_text are both NA or where sampled_text = ERROR: could not fetch page. 

The dataframe also contains features of candidates that have been combined from the camplinks.db, allowing for easier future analysis using only one CSV

In [60]:
df_filtered = data[~(data['unprocessed_text'].isna() & data['sampled_text'].isna())]
df_cleaned = df_filtered[df_filtered['sampled_text'] != 'ERROR: could not fetch page']

In [64]:
#Find the average word count for sampled_text
avg_word_count = df_cleaned['sampled_text'].str.split().str.len().mean()
print(avg_word_count)

257.6394230769231


### Running AI Text Detection ###

In [ ]:
#add in empty columns so that we don't accidently lose any pangram data
#instead, automatically append an entire row once its done; additionally run in batches 

In [101]:
#code source from pangram: https://pangram.readthedocs.io/en/latest/api/pangram.html

def check_AI(text):
    try:
        result = pangram_client.predict(text)
        valid_API_call = True
    except:
        fraction_ai = None
        fraction_ai_assisted = None
        fraction_human = None
        num_ai_segments = None
        label_final = None
        confidence_final = None

    if valid_API_call:
        fraction_ai = result['fraction_ai']
        fraction_ai_assisted = result['fraction_ai_assisted']
        fraction_human = result['fraction_human']
        num_ai_segments = result['num_ai_segments']
        label_final = result['prediction_short']
        confidence_all = []

        #for some reason, confidence is only per-window. So to get it we need to average out window confidence. Though, there tends
        #to be only 1 or 2 windows so not big deal
        for window in result['windows']:
            confidence = window['confidence']
            confidence_all.append(confidence)
            
        confidence_final = Counter(confidence_all).most_common(1)[0][0]
    
    return [label_final, confidence_final, fraction_ai, fraction_ai_assisted, fraction_human, num_ai_segments]

In [103]:
check_AI_v2("Climate policy remains one of the most pressing challenges facing governments today. Balancing economic growth with environmental protection requires difficult trade-offs, especially for nations reliant on fossil fuels. Policymakers must consider how to reduce emissions while maintaining energy affordability and job stability. Transitioning to renewable energy sources demands significant investment in infrastructure, innovation, and workforce retraining. Additionally, international cooperation is essential, as climate change transcends national borders. Without coordinated action, progress may remain uneven and insufficient. Effective climate policy must therefore integrate scientific evidence, economic realities, and social equity to ensure a sustainable and inclusive future for all citizens.")

[1.0, 0.0, 0.0, 1, 'AI', 'High']